In [34]:
import pandas as pd , numpy as np


In [35]:
df = pd.read_csv('/content/bank_marketing_dataset.csv')

df

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,subscribed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [37]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [38]:
print((df == "unknown").sum())

age                  0
job                330
marital             80
education         1731
default           8597
housing            990
loan               990
contact              0
month                0
day_of_week          0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp.var.rate         0
cons.price.idx       0
cons.conf.idx        0
euribor3m            0
nr.employed          0
subscribed           0
dtype: int64


In [39]:
print(df.shape)

(41188, 21)


In [ ]:
The model ranks customers by predicted probability of subscribing to a term deposit. The campaign manager calls the top-k customers per day, with k fixed by call-centre capacity. Costs are asymmetric: a missed subscriber loses a deposit, and a wasted call costs agent time. Performance is therefore judged by ranking quality (PR-AUC, precision@k), not accuracy. Features are limited to those available before the call, so duration is excluded as post-call leakage

#Missing values
default   : 8,597, education 1,731, housing 990, loan 990, job 330 and marital 80.

In [40]:
#subscribed is the target (y)
#calculates the share of customers who said “yes” to the term deposit.

rate = (df["subscribed"] == "yes").mean()
print(f"Positive rate: {rate:.2%}")

Positive rate: 11.27%


In [41]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 12


In [42]:
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)   # (41176, 21)

(41176, 21)


#R2 R2 — The order of operations (must have)

In [44]:
#Libraries
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

In [45]:
df = df.replace("unknown", np.nan)
y = (df["subscribed"] == "yes").astype(int)
X = df.drop(columns=["subscribed", "duration"])
print(X.shape)   # expect (41188, 19)

(41176, 19)


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)
print(f"Train positive rate: {y_train.mean():.2%} | Test: {y_test.mean():.2%}")

(32940, 19) (8236, 19)
Train positive rate: 11.27% | Test: 11.27%


In [47]:
cat = X_train.select_dtypes(include=["object", "string"]).columns
num = X_train.select_dtypes(exclude=["object", "string"]).columns
print(len(cat), len(num))   # expect 10 and 9

10 9


In [48]:
pre = ColumnTransformer([
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat),
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), num),
])

pipe = Pipeline([
    ("pre", pre),
    ("clf", HistGradientBoostingClassifier(random_state=42)),
])

In [49]:
pipe.fit(X_train, y_train)
print("Pipeline fitted on training data.")

Pipeline fitted on training data.


In [50]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

cv = TimeSeriesSplit(n_splits=5)

param_dist = {
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__max_leaf_nodes": [15, 31, 63],
    "clf__min_samples_leaf": [20, 50, 100],
    "clf__l2_regularization": [0.0, 0.1, 1.0],
}

search = RandomizedSearchCV(pipe, param_dist, n_iter=20,
                            scoring="average_precision",
                            cv=cv, random_state=42, n_jobs=-1)
search.fit(X_train, y_train)          # training data only

print("Best params:", search.best_params_)
print(f"CV PR-AUC: {search.best_score_:.3f}  (base rate ≈ {y_train.mean():.3f})")

Best params: {'clf__min_samples_leaf': 50, 'clf__max_leaf_nodes': 15, 'clf__learning_rate': 0.03, 'clf__l2_regularization': 1.0}
CV PR-AUC: 0.468  (base rate ≈ 0.113)


In [51]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

dummy = DummyClassifier(strategy="prior")
dummy_ap = cross_val_score(dummy, X_train, y_train, cv=cv,
                           scoring="average_precision").mean()
print(f"Dummy AP (baseline to beat): {dummy_ap:.3f}")
print(f"Training base rate: {y_train.mean():.3f}")

Dummy AP (baseline to beat): 0.114
Training base rate: 0.113


In [52]:
def precision_recall_at_k(y_true, scores, frac=0.10):
    """Precision and recall when calling the top `frac` of customers by score."""
    y_true = np.asarray(y_true)
    k = int(len(scores) * frac)
    top = np.argsort(scores)[::-1][:k]
    hits = y_true[top].sum()
    return hits / k, hits / y_true.sum()

In [53]:
# ===== TEST SET FIRST (AND ONLY) USE =====
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

p_test = search.best_estimator_.predict_proba(X_test)[:, 1]
p_at_k, r_at_k = precision_recall_at_k(y_test, p_test, frac=0.10)

print(f"AP (primary):  {average_precision_score(y_test, p_test):.3f}  | test base rate {y_test.mean():.3f}")
print(f"Precision@10%: {p_at_k:.3f}")
print(f"Recall@10%:    {r_at_k:.3f}")
print(f"ROC-AUC:       {roc_auc_score(y_test, p_test):.3f}")
print(f"Brier:         {brier_score_loss(y_test, p_test):.3f}")

AP (primary):  0.491  | test base rate 0.113
Precision@10%: 0.519
Recall@10%:    0.460
ROC-AUC:       0.814
Brier:         0.076


In [54]:
from scipy.stats import loguniform, randint
from sklearn.linear_model import LogisticRegression
import pandas as pd

def make_pipe(clf):
    pre = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), num),
    ])
    return Pipeline([("pre", pre), ("clf", clf)])

# Preprocessing choices tuned as hyperparameters (shared by both families)
pre_space = {
    "pre__cat__imp__strategy": ["most_frequent", "constant"],  # mode vs separate "missing" category
    "pre__num__sc": [StandardScaler(), "passthrough"],          # scale or not
}

families = {
    "LogReg": (make_pipe(LogisticRegression(max_iter=2000)),
               {**pre_space,
                "clf__C": loguniform(1e-3, 1e2)}),
    "HGB": (make_pipe(HistGradientBoostingClassifier(random_state=42)),
            {**pre_space,
             "clf__learning_rate": loguniform(1e-2, 3e-1),
             "clf__l2_regularization": loguniform(1e-3, 1e1),
             "clf__max_leaf_nodes": randint(7, 64),
             "clf__min_samples_leaf": randint(20, 200)}),
}

searches = {}
for name, (p, space) in families.items():
    s = RandomizedSearchCV(p, space, n_iter=30, scoring="average_precision",
                           cv=cv, random_state=42, n_jobs=-1)
    s.fit(X_train, y_train)          # training data only
    searches[name] = s
    print(f"{name}: best CV AP = {s.best_score_:.3f}")

LogReg: best CV AP = 0.453
HGB: best CV AP = 0.466


In [56]:
n_splits = cv.get_n_splits()

for name, s in searches.items():
    res = pd.DataFrame(s.cv_results_)
    res["se"] = res["std_test_score"] / np.sqrt(n_splits)
    res = res.sort_values("rank_test_score")

    cols = [c for c in res.columns if c.startswith("param_")] + \
           ["mean_test_score", "std_test_score", "se"]
    print(f"\n===== {name}: all {len(res)} candidates =====")
    display(res[cols].round(4))

    best = res.iloc[0]
    within = (res["mean_test_score"] >= best["mean_test_score"] - best["se"]).sum()
    print(f"{name}: {within} of {len(res)} configs within 1 SE of the best "
          f"({best['mean_test_score']:.3f} ± {best['se']:.3f})")


===== LogReg: all 30 candidates =====


,param_clf__C,param_pre__cat__imp__strategy,param_pre__num__sc,mean_test_score,std_test_score,se
1,4.5706,most_frequent,StandardScaler(),0.4529,0.0181,0.0081
27,0.9761,constant,StandardScaler(),0.4529,0.0186,0.0083
4,1.0129,constant,StandardScaler(),0.4528,0.0185,0.0083
24,0.5414,constant,StandardScaler(),0.4524,0.0189,0.0084
14,0.9164,most_frequent,StandardScaler(),0.4524,0.0187,0.0083
15,1.0907,most_frequent,StandardScaler(),0.4522,0.0188,0.0084
9,0.1445,most_frequent,StandardScaler(),0.4510,0.0190,0.0085
0,0.0746,most_frequent,StandardScaler(),0.4504,0.0190,0.0085
8,0.0332,constant,StandardScaler(),0.4491,0.0191,0.0085
11,0.0289,constant,StandardScaler(),0.4489,0.0193,0.0086


LogReg: 22 of 30 configs within 1 SE of the best (0.453 ± 0.008)

===== HGB: all 30 candidates =====


,param_clf__l2_regularization,param_clf__learning_rate,param_clf__max_leaf_nodes,param_clf__min_samples_leaf,param_pre__cat__imp__strategy,param_pre__num__sc,mean_test_score,std_test_score,se
21,0.0030,0.1131,7,24,constant,StandardScaler(),0.4656,0.0195,0.0087
22,1.2130,0.0536,15,34,constant,passthrough,0.4655,0.0164,0.0073
15,0.2680,0.0256,35,34,most_frequent,StandardScaler(),0.4639,0.0201,0.0090
7,0.5263,0.0463,24,151,most_frequent,passthrough,0.4631,0.0158,0.0071
18,0.0020,0.0338,47,47,most_frequent,StandardScaler(),0.4627,0.0206,0.0092
27,0.0075,0.0412,17,147,constant,StandardScaler(),0.4627,0.0160,0.0072
20,0.4610,0.0747,28,54,most_frequent,StandardScaler(),0.4625,0.0195,0.0087
28,3.8403,0.0295,39,148,most_frequent,StandardScaler(),0.4617,0.0152,0.0068
8,0.1791,0.0371,32,72,constant,passthrough,0.4612,0.0165,0.0074
16,0.0012,0.0422,7,155,constant,StandardScaler(),0.4610,0.0152,0.0068


HGB: 19 of 30 configs within 1 SE of the best (0.466 ± 0.009)


In [57]:
def fold_scores(s):
    i = s.best_index_
    return np.array([s.cv_results_[f"split{k}_test_score"][i] for k in range(n_splits)])

hgb, lr = fold_scores(searches["HGB"]), fold_scores(searches["LogReg"])
diff = hgb - lr

print("HGB per fold:   ", np.round(hgb, 3))
print("LogReg per fold:", np.round(lr, 3))
print("Difference:     ", np.round(diff, 3))
print(f"Mean gap: {diff.mean():.3f} | fold-to-fold std of gap: {diff.std(ddof=1):.3f} "
      f"| SE of gap: {diff.std(ddof=1)/np.sqrt(n_splits):.3f}")

HGB per fold:    [0.452 0.482 0.439 0.493 0.463]
LogReg per fold: [0.445 0.472 0.428 0.475 0.444]
Difference:      [0.007 0.009 0.01  0.018 0.019]
Mean gap: 0.013 | fold-to-fold std of gap: 0.005 | SE of gap: 0.002


In [62]:
best_model = searches["HGB"].best_estimator_
p_test = best_model.predict_proba(X_test)[:, 1]


In [63]:
s = searches["HGB"]
i = s.best_index_
cv_folds = np.array([s.cv_results_[f"split{k}_test_score"][i] for k in range(n_splits)])
cv_mean, cv_std = cv_folds.mean(), cv_folds.std(ddof=1)

print(f"CV AP (best HGB): {cv_mean:.3f} ± {cv_std:.3f}  "
      f"(std over {n_splits} folds, range {cv_folds.min():.3f}–{cv_folds.max():.3f})")

CV AP (best HGB): 0.466 ± 0.022  (std over 5 folds, range 0.439–0.493)


In [64]:

from sklearn.metrics import (average_precision_score, roc_auc_score,
                             brier_score_loss, confusion_matrix)

best_model = searches["HGB"].best_estimator_
p_test = best_model.predict_proba(X_test)[:, 1]
y_true = np.asarray(y_test)

# --- 1. Test score vs CV estimate ---
test_ap = average_precision_score(y_true, p_test)
gap = test_ap - cv_mean
print(f"CV AP:   {cv_mean:.3f} ± {cv_std:.3f}")
print(f"Test AP: {test_ap:.3f}  (test base rate {y_true.mean():.3f})")
print(f"Gap (test − CV): {gap:+.3f}  = {gap/cv_std:+.1f} CV standard deviations\n")

# --- 2. Secondary metrics (R3) ---
p_at_k, r_at_k = precision_recall_at_k(y_true, p_test, frac=0.10)
print(f"Precision@10%: {p_at_k:.3f} | Recall@10%: {r_at_k:.3f} | "
      f"ROC-AUC: {roc_auc_score(y_true, p_test):.3f} | Brier: {brier_score_loss(y_true, p_test):.3f}\n")

# --- 3. Confusion matrix at the operating point: call the top 10% ---
order = np.argsort(p_test)[::-1]
k = int(len(p_test) * 0.10)
y_pred = np.zeros_like(y_true)
y_pred[order[:k]] = 1
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
cm = pd.DataFrame([[tn, fp], [fn, tp]],
                  index=["Would NOT subscribe", "Would subscribe"],
                  columns=["Not called", "Called"])
print("Confusion matrix — operating point: call top 10%")
display(cm)

# --- 4. Capacity table ---
total_pos = y_true.sum()
rows = []
for b in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    k = int(len(p_test) * b)
    found = int(y_true[order[:k]].sum())
    rows.append({
        "Call budget": f"top {b:.0%}",
        "Calls made": k,
        "Subscribers found": found,
        "Wasted calls": k - found,
        "Calls per subscriber": round(k / found, 1),
        "% of all subscribers reached": f"{found / total_pos:.0%}",
        "Found by random calling": int(round(k * y_true.mean())),
    })
cap = pd.DataFrame(rows)
print(f"Capacity table ({len(p_test):,} customers, {total_pos:,} would subscribe)")
display(cap)

CV AP:   0.466 ± 0.022
Test AP: 0.484  (test base rate 0.113)
Gap (test − CV): +0.018  = +0.8 CV standard deviations

Precision@10%: 0.519 | Recall@10%: 0.460 | ROC-AUC: 0.809 | Brier: 0.076

Confusion matrix — operating point: call top 10%


,Not called,Called
Would NOT subscribe,6912,396
Would subscribe,501,427


Capacity table (8,236 customers, 928 would subscribe)


,Call budget,Calls made,Subscribers found,Wasted calls,Calls per subscriber,% of all subscribers reached,Found by random calling
0,top 5%,411,253,158,1.6,27%,46
1,top 10%,823,427,396,1.9,46%,93
2,top 15%,1235,573,662,2.2,62%,139
3,top 20%,1647,614,1033,2.7,66%,186
4,top 30%,2470,681,1789,3.6,73%,278
5,top 50%,4118,775,3343,5.3,84%,464


In [65]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate

rows = []
for strat in ["prior", "most_frequent", "stratified", "uniform", "constant"]:
    d = DummyClassifier(strategy=strat, constant=1 if strat == "constant" else None, random_state=42)
    r = cross_validate(d, X_train, y_train, cv=cv, scoring=["average_precision", "accuracy"])
    rows.append({"dummy": strat,
                 "CV AP": r["test_average_precision"].mean(),
                 "CV accuracy": r["test_accuracy"].mean()})
base = pd.DataFrame(rows)
base["AP margin of model"] = cv_mean - base["CV AP"]
display(base.round(3))

,dummy,CV AP,CV accuracy,AP margin of model
0,prior,0.114,0.886,0.352
1,most_frequent,0.114,0.886,0.352
2,stratified,0.114,0.802,0.352
3,uniform,0.114,0.500,0.352
4,constant,0.114,0.114,0.352


In [66]:
from sklearn.model_selection import cross_val_score

single = []
for col in X_train.columns:
    if col in cat:
        pre1 = ColumnTransformer([("c", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), [col])])
    else:
        pre1 = ColumnTransformer([("n", SimpleImputer(strategy="median"), [col])])
    p1 = Pipeline([("pre", pre1), ("clf", HistGradientBoostingClassifier(random_state=42))])
    ap = cross_val_score(p1, X_train, y_train, cv=cv,
                         scoring="average_precision", n_jobs=-1).mean()
    single.append((col, ap))

single = pd.DataFrame(single, columns=["feature", "CV AP"]).sort_values("CV AP", ascending=False)
single["% of full model"] = (single["CV AP"] / cv_mean * 100).round(0)
display(single.round(3))

,feature,CV AP,% of full model
17,euribor3m,0.375,81.0
15,cons.price.idx,0.366,79.0
16,cons.conf.idx,0.365,78.0
18,nr.employed,0.342,74.0
14,emp.var.rate,0.310,67.0
13,poutcome,0.241,52.0
8,month,0.235,50.0
11,pdays,0.233,50.0
0,age,0.198,43.0
12,previous,0.195,42.0


In [67]:
from sklearn.base import clone

seed_scores = []
for sd in [0, 1, 2, 3, 4]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=sd)
    m = clone(best_model).fit(Xtr, ytr)
    seed_scores.append(average_precision_score(yte, m.predict_proba(Xte)[:, 1]))

seed_scores = np.array(seed_scores)
mu, sd_ = seed_scores.mean(), seed_scores.std(ddof=1)
print("AP per seed:", np.round(seed_scores, 3))
print(f"Mean {mu:.3f} ± {sd_:.3f} | reported test AP {test_ap:.3f} sits "
      f"{(test_ap - mu) / sd_:+.1f} SD from the mean")

AP per seed: [0.462 0.473 0.479 0.463 0.483]
Mean 0.472 ± 0.009 | reported test AP 0.484 sits +1.2 SD from the mean


In [68]:
from sklearn.base import clone

seed_scores = []
for sd in [0, 1, 2, 3, 4]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=sd)
    m = clone(best_model).fit(Xtr, ytr)
    seed_scores.append(average_precision_score(yte, m.predict_proba(Xte)[:, 1]))

seed_scores = np.array(seed_scores)
mu, sd_ = seed_scores.mean(), seed_scores.std(ddof=1)
print("AP per seed:", np.round(seed_scores, 3))
print(f"Mean {mu:.3f} ± {sd_:.3f} | reported test AP {test_ap:.3f} sits "
      f"{(test_ap - mu) / sd_:+.1f} SD from the mean")

AP per seed: [0.462 0.473 0.479 0.463 0.483]
Mean 0.472 ± 0.009 | reported test AP 0.484 sits +1.2 SD from the mean


In [69]:
rng = np.random.default_rng(0)
y_perm = pd.Series(rng.permutation(y_train.values), index=y_train.index)

perm = {}
for name, (p, space) in families.items():
    s_perm = RandomizedSearchCV(p, space, n_iter=30, scoring="average_precision",
                                cv=cv, random_state=42, n_jobs=-1)
    s_perm.fit(X_train, y_perm)
    perm[name] = s_perm.best_score_
    print(f"{name} on shuffled labels: best CV AP = {s_perm.best_score_:.3f} "
          f"(baseline {y_train.mean():.3f})")

LogReg on shuffled labels: best CV AP = 0.117 (baseline 0.113)
HGB on shuffled labels: best CV AP = 0.116 (baseline 0.113)
